# 正常情境擴充與重新平衡訓練
訓練 127,455／驗證 25,592／舊來源診斷測試 25,102。使用 mirage-diverse-training-v2.zip。
開發正常流程與封存驗收流程分開；此筆記本不含獨立驗收資料，不能用合併高分宣稱部署驗收通過。
8,000 是 TF-IDF 特徵上限，使用 CPU。重現新模型後需另行驗收；不要直接開啟自動分流。


## 1. 掛載 Drive、複製並解壓到本機磁碟

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys, zipfile

DRIVE = Path('/content/drive/MyDrive/MirageSentinel')
SOURCE_ZIP = DRIVE / 'mirage-diverse-training-v2.zip'
assert SOURCE_ZIP.is_file(), f'請上傳資料包到 {SOURCE_ZIP}'
with SOURCE_ZIP.open('rb') as stream:
    actual = hashlib.file_digest(stream, 'sha256').hexdigest()
assert actual == '5960054f5a926f8bf67b8ea6293621ad106a2cd55b47484186c0b28bc96fdf74', '資料包版本或內容不符，請使用搭配此筆記本的新 ZIP'
WORK = Path('/content/mirage-diverse-v2-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
WORK.mkdir(exist_ok=False)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for member in archive.infolist():
        assert (WORK / member.filename).resolve().is_relative_to(WORK.resolve()), 'ZIP 路徑不安全'
    archive.extractall(WORK)
for name, expected in json.loads((WORK / 'package-checksums.json').read_text()).items():
    path = (WORK / name).resolve()
    assert path.is_relative_to(WORK.resolve())
    with path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, name
DATA = WORK / 'data'
manifest = json.loads((DATA / 'manifest.json').read_text())
print('新資料包與全部檔案校驗通過。工作目錄：', WORK)

## 2. 安裝訓練環境
保留 virtualenv 修正，安裝與檢查過程會即時顯示。

In [ ]:
import os
print(sys.version)
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
print(f'系統 RAM：{ram_gib:.1f} GiB；CPU：{os.cpu_count()}')
assert (3, 11) <= sys.version_info[:2] <= (3, 13), '此固定套件組合支援 Python 3.11～3.13'
if ram_gib < 24:
    print('提醒：此程式不是外部記憶體訓練；目前 RAM 可能不足，無法保證完成。')
def run_visible(command: list[str], cwd: Path | None = None) -> None:
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('步驟失敗，請查看上方完整輸出：' + ' '.join(command))

# Some hosted Python builds lack the ensurepip support required by stdlib venv.
# virtualenv supplies its own pip seed; leave the incomplete old directory alone.
run_visible([sys.executable, '-m', 'pip', 'install', 'virtualenv'])
VENV = WORK / 'venv'
run_visible([sys.executable, '-m', 'virtualenv', '--python', sys.executable, str(VENV)])
PYTHON = str(VENV / 'bin/python')
run_visible([PYTHON, '-m', 'pip', 'install', '-r', str(WORK / 'commerce_ml/requirements.txt')])
run_visible([PYTHON, '-m', 'pip', 'check'])
run_visible([PYTHON, '-c', 'import sys, numpy, scipy, sklearn, xgboost; print(sys.version); print(numpy.__version__, scipy.__version__, sklearn.__version__, xgboost.__version__)'])
run_visible([PYTHON, '-m', 'commerce_ml.train', '--help'], cwd=WORK)

## 3. 設定這次實驗
開始前決定參數。執行訓練後會自動評估測試集；不要看完測試結果再用同一測試集反覆調參。

In [ ]:
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
RUN = RUNS / ('run-' + STAMP)
AUDIT = RUNS / ('audit-' + STAMP)
BACKUPS = DRIVE / 'runs'
BACKUPS.mkdir(parents=True, exist_ok=True)
ROUNDS = 600
MAX_FEATURES = 8000
THREADS = min(2, os.cpu_count() or 1)
COMMON = [PYTHON, '-u', '-m', 'commerce_ml.train',
          '--train', str(DATA / 'train'),
          '--validation', str(DATA / 'validation'),
          '--test', str(DATA / 'test'),
          '--allow-unreviewed', '--allow-shared-families', '--allow-shared-normal-templates', '--balance-normal-routes', '--early-stopping-metric', 'mlogloss']
print('本次輸出：', RUN)

## 4. 資料稽核
看到「Audit passed; no training performed.」與「178,149 筆資料稽核通過」後才繼續。

In [ ]:
run_visible(COMMON + ['--output', str(AUDIT), '--audit-only'], cwd=WORK)
audit = json.loads((AUDIT / 'data-audit.json').read_text())
assert not audit['cross_split_conflicts']
assert {k: v['rows'] for k, v in audit['splits'].items()} == {'train': 127455, 'validation': 25592, 'test': 25102}
for split, info in audit['splits'].items():
    assert info['labels'] == manifest['splits'][split]['labels']
    assert info['exact_duplicates'] == 0
print('178,149 筆資料稽核通過；尚未開始訓練。')

## 5. 正式訓練與評估
使用完整新訓練集，TF-IDF 和權重只在訓練集計算；驗證集選擇停止輪數及門檻。
此程式固定 CPU hist。`batch-size` 不是完整記憶體上限。
訓練完成或一般程序失敗時保存 log 並備份成果；若整個 Colab VM 被回收，備份區塊可能無法執行。
目前沒有 checkpoint 續訓功能。失敗後先處理原因，重跑第 3 格產生新輸出路徑。


In [ ]:
assert (AUDIT / 'data-audit.json').is_file(), '請先完成第 4 步'
LOG = RUNS / ('training-' + STAMP + '.log')
command = COMMON + ['--output', str(RUN), '--threads', str(THREADS),
                    '--rounds', str(ROUNDS), '--early-stopping', '30',
                    '--max-features', str(MAX_FEATURES), '--batch-size', '2048',
                    '--seed', '42', '--max-fpr', '0.01']
try:
    with LOG.open('w') as log:
        with subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'訓練失敗，exit code={returncode}；請檢查 log，勿把部分輸出當成成功模型。')
finally:
    if LOG.exists():
        shutil.copy2(LOG, BACKUPS / LOG.name)
    if RUN.exists():
        archive_path = shutil.make_archive(str(RUN), 'zip', root_dir=RUN.parent, base_dir=RUN.name)
        shutil.copy2(archive_path, BACKUPS / Path(archive_path).name)
        print('成果／中間產物已備份：', BACKUPS / Path(archive_path).name)

## 6. 驗證模型包、分來源評估並備份
訓練器的合併分數只供參考。此步驟對 validation 和 test 各來源分開評估，不重新訓練或調閾值。
synthetic_context 是合成上下文診斷；單一 XSS 來源不能估計正常誤報率或完整五類表現。
若此步驟失敗，可修正原因後重跑；輸出放在新的時間戳記目錄。

In [ ]:
assert (RUN / 'checksums.json').is_file(), '訓練尚未完整完成'
for name, expected in json.loads((RUN / 'checksums.json').read_text()).items():
    with (RUN / name).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, name
run_visible([PYTHON, str(RUN / 'bundle/runtime.py'), '--self-test'], cwd=WORK)
EVAL = RUN / ('by-source-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
EVAL.mkdir()
try:
    for split in ('validation', 'test'):
        run_visible([PYTHON, '-u', '-m', 'commerce_ml.evaluate_context',
                     '--bundle', str(RUN / 'bundle'), '--data', str(DATA),
                     '--split', split, '--output', str(EVAL / split)], cwd=WORK)
    for path in sorted(EVAL.glob('*/*-metrics.json')):
        metrics = json.loads(path.read_text())
        print(path.parent.name, path.stem, json.dumps({
            'support': metrics['support'], 'accuracy': metrics['accuracy'],
            'macro_f1_all_five': metrics['macro_f1_all_five'],
            'per_class_report': metrics['per_class_report'], 'confusion_matrix': metrics['confusion_matrix'], 'binary_comparison': metrics['binary_comparison']}, ensure_ascii=False))
    print('分來源評估完成；合成來源成績不可當成真實攻擊偵測率。')
finally:
    archive_path = shutil.make_archive(str(RUN), 'zip', root_dir=RUN.parent, base_dir=RUN.name)
    shutil.copy2(archive_path, BACKUPS / Path(archive_path).name)
    print('包含分來源結果的備份：', BACKUPS / Path(archive_path).name)

## 7. 部署限制
服務端已接上五分類觀察模式。這份試訓包仍需新的正常購物流程驗收；不要直接覆蓋已部署模型或開啟自動分流。
